### ЗАДАЧА: Пакетная загрузка конфигов деплоя

От DevOps-команды приходит пакет строк с конфигами сервисов для выкладки.
Нужно обработать их так, чтобы:
- валидные конфиги попали в итоговый список,
- проблемные записи не остановили весь пакет,
- по ошибкам собрался отдельный журнал,
- в конце было видно, какие сервисы включены по окружениям и какой у них средний timeout.

Часть строк содержит ошибки в формате и числах,
часть использует неизвестное окружение или неправильный флаг включения.


In [1]:
# service|max_retries|timeout_sec|environment|enabled
rows = [
    'auth|3|1.5|prod|on',
    'billing|0|2.0|stage|on',
    'search|two|0.8|dev|off',
    'media|5|-1|prod|on',
    'chat|4|1.2|test|off',
    'mail|2|0.5|stage|maybe',
    'worker|1|3.4|prod|on',
]


class DeployConfigError(Exception):
    pass


class RowFormatError(DeployConfigError):
    pass


class RetriesError(DeployConfigError):
    pass


class TimeoutError(DeployConfigError):
    pass


class EnvironmentError(DeployConfigError):
    pass


class EnabledFlagError(DeployConfigError):
    pass


def parse_config(row):
    # TODO: распарсить строку и провалидировать max_retries, timeout_sec, environment, enabled
    # TODO: при ошибках конвертации использовать raise ... from ...
    # TODO: enabled вернуть как bool
    parts = row.split("|")
    if len(parts) != 5:
        raise RowFormatError(f"Ошибка формата страки {row}")
    
    service, max_retries_str, timeout_str, environment, enabled = parts

    try:
        max_retries = int(max_retries_str)
        if max_retries < 0:
            raise RetriesError(f"max_retries не может быть отрецательным {max_retries} в строке {row}")
    except ValueError as e:
        raise RetriesError(f"Невалидное число max_retries: {max_retries_str}") from e
    
    try:
        timeout_sec = float(timeout_str)
        if timeout_sec <= 0:
            raise TimeoutError(f"timeout_sec должен быть положительным: {timeout_sec} в строке: {row}")
    except ValueError as e:
        raise TimeoutError(f"Не валидное число timeout_sec: {timeout_str}") from e
    
    environment = environment.strip()
    if environment not in ("prod", "stage", "dev", "test"):
        raise EnabledFlagError(f"Неизвестное окружение: {environment}")
    
    if enabled == "on":
        enabled = True
    elif enabled == "off":
        enabled = False
    else:
        raise EnabledFlagError(f"Некорректное значение enabled: {enabled} в строке: {row}")
    
    return{
        "service": service,
        "max_retries": max_retries,
        "timeout_sec": timeout_sec,
        "environment": environment,
        "enabled": enabled
    }

def load_configs(rows):
    # TODO: вернуть (configs, errors)
    # TODO: вызвать load_configs(rows)
    configs = []
    errors = []

    for row in rows:
        try:
            conf = parse_config(row)
            configs.append(conf)
        except DeployConfigError as e:
            errors.append({"row": row, "error": str(e)})
    
    return configs, errors

configs, errors = load_configs(rows)


# TODO: вывести число валидных конфигов и число ошибок
print(f"Всего валидных конфигов: {len(configs)}")
print(f"Кол-во ошибок: {len(errors)}")

# TODO: вывести ошибки по типам
error_types = {}
for  err in errors:
    err_msg = err
    err_type = type(err_msg).__name__
    error_types[err_type] = error_types.get(err_type, 0) + 1
print("Ошибка")

for etype, count in error_types.items():
    print (f"{etype}: {count}")

# TODO: собрать enabled_by_environment: dict[str, list[str]]
enabled_by_environment = {}
for conf in configs:
    env = conf["environment"]
    if env not in enabled_by_environment:
        enabled_by_environment[env] = []
    status = "enabled" if conf["enabled"] else "disabled"
    enabled_by_environment[env].append(f"{conf["service"]} ({status})")

    print("Сервесы по окрыжениям:")
    for env, services in enabled_by_environment.items():
        print(f"{env}: {",".join(services)}")

# TODO: посчитать average_timeout только по enabled=True
enabled_configs = [c for c in configs if c ["enabled"]]
if enabled_configs:
    avg_timeout = sum(c["timeout_sec"] for c in enabled_configs) / len(enabled_configs)
    print(f"Средний тамаут включенных серверов: {avg_timeout:.2f} сек")
else:
    print("Нет включенных серверов для расчёта среднего таймаута")

Всего валидных конфигов: 4
Кол-во ошибок: 3
Ошибка
dict: 3
Сервесы по окрыжениям:
prod: auth (enabled)
Сервесы по окрыжениям:
prod: auth (enabled)
stage: billing (enabled)
Сервесы по окрыжениям:
prod: auth (enabled)
stage: billing (enabled)
test: chat (disabled)
Сервесы по окрыжениям:
prod: auth (enabled),worker (enabled)
stage: billing (enabled)
test: chat (disabled)
Средний тамаут включенных серверов: 2.30 сек
